# 00 · Recolectar noticias desde NewsData

Este notebook reúne la consulta a NewsData, la extracción de cuerpos de noticias y la preparación local de tareas para Label Studio. Usa BeautifulSoup para extraer HTML.

Alcance de este paso: inventario, extracción técnica de cuerpos y tareas sin anotación. No extrae claims ni evalúa veracidad.

## 1. Cargar la clave local

En la raíz del repositorio crea `.env` con una sola línea: `NEWSDATA_API_KEY=tu_clave`. El notebook lee esa clave sin mostrarla ni guardarla en sus salidas.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file() and (ROOT.parent / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Abre el notebook desde la raíz del repositorio o desde su carpeta notebooks/.')

def load_local_env(path: Path) -> None:
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('export '):
            line = line.removeprefix('export ').strip()
        if '=' not in line:
            continue
        name, value = line.split('=', 1)
        name, value = name.strip(), value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {chr(34), chr(39)}:
            value = value[1:-1]
        if name:
            os.environ.setdefault(name, value)

dotenv_path = ROOT / '.env'
if not dotenv_path.is_file():
    raise FileNotFoundError('Falta .env en la raíz. Agrega NEWSDATA_API_KEY=tu_clave.')
load_local_env(dotenv_path)
API_KEY = os.environ.get('NEWSDATA_API_KEY', '').strip()
if not API_KEY:
    raise RuntimeError('NEWSDATA_API_KEY no está definida en .env.')

print('Entorno listo; la clave fue detectada y no se mostrará.')

## 2. Definir una corrida LATAM equilibrada

La selección final apunta a 50 tareas con cuota estricta. Consulta candidatos de los 19 países/territorios LATAM y, después de extraer los cuerpos, conserva 2 o 3 tareas válidas por país. Si un país no llega a su cuota, el notebook reporta el déficit y no crea el CSV para Label Studio.

In [ ]:
LATAM_COUNTRIES = [
    'ar', 'bo', 'cl', 'co', 'cr', 'cu', 'do', 'ec', 'gt', 'hn',
    'mx', 'ni', 'pa', 'pe', 'pr', 'py', 'sv', 'uy', 've',
]
import random

TARGET_NEWS = 50
QUOTA_SEED = 20261103  # hace reproducible qué 12 países reciben una tercera tarea
QUERY = '(salud OR medicina OR médico OR enfermedad OR vacuna OR tratamiento)'
RESULTS_PER_COUNTRY = 10  # candidatos por país antes de aplicar round robin
MAX_PAGES_PER_COUNTRY = 1
TIMEOUT_SECONDS = 30
DELAY_SECONDS = 1.0

countries = list(LATAM_COUNTRIES)
if len(countries) != len(set(countries)) or any(len(country) != 2 or not country.isalpha() for country in countries):
    raise ValueError('LATAM_COUNTRIES debe contener códigos ISO-2 únicos.')
if not (TARGET_NEWS >= len(countries) and 1 <= RESULTS_PER_COUNTRY <= 10 and MAX_PAGES_PER_COUNTRY >= 1 and TIMEOUT_SECONDS > 0 and DELAY_SECONDS >= 0):
    raise ValueError('Revisa TARGET_NEWS, RESULTS_PER_COUNTRY y los límites de solicitud.')

# La cuota se calcula antes de consultar la API para repartir la muestra entre países.
base_quota, extra_slots = divmod(TARGET_NEWS, len(countries))
quota_rng = random.Random(QUOTA_SEED)
extra_countries = set(quota_rng.sample(countries, extra_slots))
country_quotas = {country: base_quota + int(country in extra_countries) for country in countries}

print(f"Corrida preparada: {TARGET_NEWS} tareas finales con cuotas de {base_quota} o {base_quota + 1} por país.")
print('Cuota 3:', ', '.join(sorted(extra_countries)))
print('Cuota 2:', ', '.join(country for country in countries if country not in extra_countries))

## 3. Funciones locales de normalización

Estas funciones viven únicamente en este notebook. Limpian URLs de parámetros de seguimiento, generan un identificador estable y conservan solo noticias en español de la categoría salud.

In [ ]:
import hashlib
import re
import unicodedata
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

TRACKING_PARAMETERS = {'fbclid', 'gclid', 'mc_cid', 'mc_eid'}
COUNTRY_ALIASES = {
    'ar': {'ar', 'argentina'}, 'bo': {'bo', 'bolivia', 'bolivia plurinational state of'},
    'cl': {'cl', 'chile'}, 'co': {'co', 'colombia'}, 'cr': {'cr', 'costa rica'},
    'cu': {'cu', 'cuba'}, 'do': {'do', 'dominican republic', 'republica dominicana'},
    'ec': {'ec', 'ecuador'}, 'gt': {'gt', 'guatemala'}, 'hn': {'hn', 'honduras'},
    'mx': {'mx', 'mexico'}, 'ni': {'ni', 'nicaragua'}, 'pa': {'pa', 'panama'},
    'pe': {'pe', 'peru'}, 'pr': {'pr', 'puerto rico'}, 'py': {'py', 'paraguay'},
    'sv': {'sv', 'el salvador'}, 'uy': {'uy', 'uruguay'}, 've': {'ve', 'venezuela'},
}
INVENTORY_COLUMNS = [
    'record_id', 'source_name', 'canonical_url', 'published_at', 'title',
    'subtitle_or_bajada', 'topic', 'selection_status', 'retrieved_at', 'requested_country', 'country',
]

def clean_text(value):
    return re.sub(r'\s+', ' ', str(value or '')).strip()

def canonicalize_url(raw_url):
    parsed = urlsplit(raw_url.strip())
    if parsed.scheme.lower() not in {'http', 'https'} or not parsed.hostname:
        raise ValueError('La URL debe usar http(s) y tener dominio.')
    query = [(key, value) for key, value in parse_qsl(parsed.query, keep_blank_values=True)
             if key.lower() not in TRACKING_PARAMETERS and not key.lower().startswith('utm_')]
    hostname = parsed.hostname.lower()
    netloc = hostname if parsed.port is None else f'{hostname}:{parsed.port}'
    return urlunsplit((parsed.scheme.lower(), netloc, parsed.path or '/', urlencode(query), ''))

def record_id_for_url(canonical_url):
    return hashlib.sha256(canonical_url.encode('utf-8')).hexdigest()[:16]

def normalize_country(value, requested_country):
    raw = ','.join(str(item) for item in value) if isinstance(value, list) else str(value or '')
    normalized = unicodedata.normalize('NFKD', raw.casefold())
    normalized = ''.join(char for char in normalized if not unicodedata.combining(char))
    parts = {part.strip() for part in re.split(r'[,;/|]', normalized) if part.strip()}
    for code, aliases in COUNTRY_ALIASES.items():
        if parts.intersection(aliases):
            return code
    return requested_country

def normalize_articles(articles, retrieved_at, requested_country):
    rows, seen_ids = [], set()
    for article in articles:
        if not isinstance(article, dict):
            continue
        language = clean_text(article.get('language')).casefold()
        categories_raw = article.get('category')
        categories = {clean_text(item).casefold() for item in categories_raw} if isinstance(categories_raw, list) else {clean_text(categories_raw).casefold()}
        if language not in {'es', 'spanish'} or 'health' not in categories:
            continue
        raw_url = clean_text(article.get('link'))
        if not raw_url:
            continue
        try:
            canonical_url = canonicalize_url(raw_url)
        except ValueError:
            continue
        record_id = record_id_for_url(canonical_url)
        if record_id in seen_ids:
            continue
        seen_ids.add(record_id)
        rows.append({
            'record_id': record_id, 'source_name': clean_text(article.get('source_name')),
            'canonical_url': canonical_url,
            'published_at': clean_text(article.get('pubDate') or article.get('pub_date')),
            'title': clean_text(article.get('title')),
            'subtitle_or_bajada': clean_text(article.get('description')),
            'topic': 'health', 'selection_status': 'PENDIENTE',
            'retrieved_at': retrieved_at, 'requested_country': requested_country,
            'country': normalize_country(article.get('country'), requested_country),
        })
    return rows

## 4. Consultar NewsData

Esta celda realiza la llamada externa. Si recibes un error, revisa la clave, cuota, países o conexión; la clave nunca se imprime.

In [ ]:
import json
import time
from datetime import datetime, timezone
from collections import Counter
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

def request_pages(country):
    articles, page = [], None
    for index in range(MAX_PAGES_PER_COUNTRY):
        if index and DELAY_SECONDS:
            time.sleep(DELAY_SECONDS)
        params = {
            'apikey': API_KEY, 'language': 'es', 'category': 'health', 'country': country,
            'q': QUERY, 'size': str(RESULTS_PER_COUNTRY), 'removeduplicate': '1', 'video': '0',
        }
        if page:
            params['page'] = page
        request = Request(
            'https://newsdata.io/api/1/latest?' + urlencode(params),
            headers={'Accept': 'application/json', 'User-Agent': 'mednews-notebook/1.0'},
            method='GET',
        )
        try:
            with urlopen(request, timeout=TIMEOUT_SECONDS) as response:
                payload = json.loads(response.read().decode('utf-8'))
        except HTTPError as exc:
            raise RuntimeError(f'NewsData devolvió HTTP {exc.code}. Revisa clave, cuota o parámetros.') from exc
        except (URLError, TimeoutError) as exc:
            raise RuntimeError('No se pudo consultar NewsData; revisa la conexión.') from exc
        if not isinstance(payload, dict) or str(payload.get('status', '')).lower() in {'error', 'failed'}:
            raise RuntimeError('NewsData devolvió una respuesta no válida; revisa clave, cuota o parámetros.')
        page_results = payload.get('results', [])
        if isinstance(page_results, list):
            articles.extend(page_results)
        page = str(payload.get('nextPage') or '').strip() or None
        if not page:
            break
    return articles

retrieved_at = datetime.now(timezone.utc).isoformat()
candidate_pools, seen_record_ids = {}, set()
for index, country in enumerate(countries):
    if index and DELAY_SECONDS:
        time.sleep(DELAY_SECONDS)
    country_candidates = []
    for candidate in normalize_articles(request_pages(country), retrieved_at, country):
        if candidate['record_id'] in seen_record_ids:
            continue
        seen_record_ids.add(candidate['record_id'])
        country_candidates.append(candidate)
    candidate_pools[country] = country_candidates
candidate_rows = [candidate for country in countries for candidate in candidate_pools[country]]
rows = candidate_rows  # conserva compatibilidad con la exportación del pool local

candidate_counts = {country: len(candidate_pools[country]) for country in countries}
candidate_deficits = {country: country_quotas[country] - candidate_counts[country] for country in countries if candidate_counts[country] < country_quotas[country]}
print(f'Candidatos únicos recuperados: {len(candidate_rows)}')
print('Candidatos por país:', ', '.join(f'{country}={candidate_counts[country]}' for country in countries))
if candidate_deficits:
    print('Aviso temprano — candidatos insuficientes para la cuota:', candidate_deficits)
for row in candidate_rows[:5]:
    print({key: row[key] for key in ('record_id', 'requested_country', 'country', 'source_name', 'published_at', 'title')})

## 5. Guardar el pool local de candidatos

Este CSV conserva todos los candidatos únicos recuperados antes de filtrar por cuerpo. Es un artefacto de trazabilidad, no el archivo que se importará en Label Studio.

In [ ]:
import csv

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
output_path = ROOT / 'artifacts' / 'collection' / run_id / 'newsdata_candidates.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=INVENTORY_COLUMNS)
    writer.writeheader()
    writer.writerows(rows)

print(f'Pool local de candidatos guardado en: {output_path.relative_to(ROOT)}')

## 6. Extraer cuerpos y preparar Label Studio

Las siguientes celdas, dentro de este mismo notebook, descargan las URL con BeautifulSoup, guardan diagnósticos de las 50 noticias y crean un CSV que se puede importar como tareas de Label Studio.

## 7. Descargar y extraer los cuerpos

Esta celda visita las URL públicas de las noticias seleccionadas. Puede tardar varios minutos. Conserva tanto los cuerpos utilizables como los errores técnicos: ninguna fila desaparece silenciosamente.

In [ ]:
try:
    from bs4 import BeautifulSoup
except ImportError as exc:
    raise RuntimeError('El kernel necesita BeautifulSoup para extraer HTML.') from exc

from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

BODY_TIMEOUT_SECONDS = 15
BODY_DELAY_SECONDS = 0.5
BODY_MAX_BYTES = 1_500_000
BODY_MAX_TEXT_CHARS = 200_000
BODY_MAX_RETRIES = 1
BODY_MIN_WORDS = 150
BODY_MIN_CHARS = 800
BODY_USER_AGENT = 'Mozilla/5.0 (compatible; mednews-thesis/1.0; research)'
BODY_SELECTORS = (
    "[itemprop='articleBody']", 'article', '.article-body', '.article__body',
    '.article-content', '.story-body', '.entry-content', '.post-content', 'main',
)

def decode_html(raw, content_type):
    match = re.search(r"charset\s*=\s*['\"]?([\w.-]+)", content_type, re.I)
    encoding = match.group(1) if match else 'utf-8'
    try:
        return raw.decode(encoding, errors='replace')
    except LookupError:
        return raw.decode('utf-8', errors='replace')

def extract_body_from_html(html):
    soup = BeautifulSoup(html, 'html.parser')
    for tag in soup.find_all(['script', 'style', 'noscript', 'template', 'svg', 'canvas', 'nav', 'header', 'footer', 'aside', 'form']):
        tag.decompose()
    candidates = []
    for selector in BODY_SELECTORS:
        for node in soup.select(selector):
            text = clean_text(node.get_text(' ', strip=True))
            if text:
                candidates.append(text)
    if not candidates:
        fallback = clean_text((soup.body or soup).get_text(' ', strip=True))
        if fallback:
            candidates.append(fallback)
    return (max(candidates, key=len), 'beautifulsoup') if candidates else ('', 'none')

def download_html(url):
    if not url.startswith(('http://', 'https://')):
        raise ValueError('URL http(s) ausente o inválida.')
    last_error = None
    for attempt in range(BODY_MAX_RETRIES + 1):
        request = Request(url, headers={
            'Accept': 'text/html,application/xhtml+xml',
            'Accept-Language': 'es-419,es;q=0.9,en;q=0.5',
            'User-Agent': BODY_USER_AGENT,
        }, method='GET')
        try:
            with urlopen(request, timeout=BODY_TIMEOUT_SECONDS) as response:
                raw = response.read(BODY_MAX_BYTES + 1)[:BODY_MAX_BYTES]
                return raw, int(response.status or 200), str(response.headers.get('Content-Type', ''))
        except HTTPError:
            raise
        except (URLError, TimeoutError, OSError) as exc:
            last_error = exc
            if attempt < BODY_MAX_RETRIES:
                time.sleep(min(2 ** (attempt + 1), 4))
    raise last_error or RuntimeError('No se pudo descargar la URL.')

def extract_body_row(source_row):
    output = dict(source_row)
    output.update({
        'body': '', 'body_available': 'NO', 'http_status': '',
        'extraction_method': 'none', 'extraction_status': 'ERROR',
        'body_char_count': 0, 'body_word_count': 0,
        'extracted_at': datetime.now(timezone.utc).isoformat(), 'extraction_error': '',
    })
    try:
        raw, http_status, content_type = download_html(source_row['canonical_url'])
        body, method = extract_body_from_html(decode_html(raw, content_type))
        body = body[:BODY_MAX_TEXT_CHARS]
        word_count = len(re.findall(r'\b[\wÀ-ÿ]+\b', body, flags=re.UNICODE))
        char_count = len(body)
        status = 'OK' if word_count >= BODY_MIN_WORDS and char_count >= BODY_MIN_CHARS else 'CUERPO_INSUFICIENTE'
        output.update({
            'body': body, 'body_available': 'SI' if status == 'OK' else 'NO',
            'http_status': http_status, 'extraction_method': method, 'extraction_status': status,
            'body_char_count': char_count, 'body_word_count': word_count,
        })
    except HTTPError as exc:
        output.update({'http_status': exc.code, 'extraction_status': 'HTTP_ERROR', 'extraction_error': f'HTTP {exc.code}'})
    except TimeoutError as exc:
        output.update({'extraction_status': 'TIMEOUT', 'extraction_error': str(exc)[:500]})
    except ValueError as exc:
        output.update({'extraction_status': 'URL_ERROR', 'extraction_error': str(exc)[:500]})
    except (URLError, OSError) as exc:
        output.update({'extraction_status': 'URL_ERROR', 'extraction_error': str(exc)[:500]})
    except Exception as exc:
        output.update({'extraction_status': 'ERROR', 'extraction_error': f'{type(exc).__name__}: {str(exc)[:450]}'})
    return output

body_rows = []
valid_by_country = {country: [] for country in countries}
attempted_by_country = {country: 0 for country in countries}
next_candidate_index = {country: 0 for country in countries}

# Una URL por país en cada ronda. Si falla, se prueba el siguiente candidato
# del mismo país hasta completar su cuota o agotar su pool.
while True:
    attempted_in_round = False
    for country in countries:
        if len(valid_by_country[country]) >= country_quotas[country]:
            continue
        pool = candidate_pools[country]
        candidate_index = next_candidate_index[country]
        if candidate_index >= len(pool):
            continue
        if body_rows and BODY_DELAY_SECONDS:
            time.sleep(BODY_DELAY_SECONDS)
        result = extract_body_row(pool[candidate_index])
        next_candidate_index[country] += 1
        attempted_by_country[country] += 1
        body_rows.append(result)
        attempted_in_round = True
        if result['extraction_status'] == 'OK':
            valid_by_country[country].append(result)
    if not attempted_in_round:
        break

balanced_rows = [
    row for country in countries for row in valid_by_country[country][:country_quotas[country]]
]
deficits = {
    country: country_quotas[country] - len(valid_by_country[country])
    for country in countries if len(valid_by_country[country]) < country_quotas[country]
}
body_statuses = Counter(row['extraction_status'] for row in body_rows)
print(f'URLs procesadas para cubrir cuotas: {len(body_rows)}')
print('Estados técnicos:', dict(sorted(body_statuses.items())))
print('Cuerpos OK por país:', ', '.join(f'{country}={len(valid_by_country[country])}/{country_quotas[country]}' for country in countries))
if deficits:
    print('Déficit: no se generarán tareas de Label Studio.', deficits)
else:
    print(f'Cuotas completas: {len(balanced_rows)} tareas balanceadas listas.')

## 8. Crear el CSV para Label Studio

Siempre se guardan `newsdata_bodies.csv` y `balance_report.csv`. Solo cuando todas las cuotas se cumplen se crea `labelstudio_tasks.csv`, con 50 tareas equilibradas y cuerpo `OK`.

In [ ]:
import csv

BODY_COLUMNS = INVENTORY_COLUMNS + [
    'body', 'body_available', 'http_status', 'extraction_method', 'extraction_status',
    'body_char_count', 'body_word_count', 'extracted_at', 'extraction_error',
]
# Este CSV contiene texto y metadatos mínimos para importar una tarea por noticia en Label Studio.
LABEL_STUDIO_COLUMNS = [
    'record_id', 'title', 'body', 'canonical_url', 'source_name', 'published_at',
    'country', 'requested_country', 'subtitle_or_bajada',
]

bodies_path = output_path.parent / 'newsdata_bodies.csv'
with bodies_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=BODY_COLUMNS, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(body_rows)

report_path = output_path.parent / 'balance_report.csv'
report_rows = [
    {
        'requested_country': country, 'quota': country_quotas[country],
        'candidates_retrieved': len(candidate_pools[country]),
        'urls_attempted': attempted_by_country[country],
        'bodies_ok': len(valid_by_country[country]),
        'deficit': max(0, country_quotas[country] - len(valid_by_country[country])),
    }
    for country in countries
]
with report_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['requested_country', 'quota', 'candidates_retrieved', 'urls_attempted', 'bodies_ok', 'deficit'])
    writer.writeheader()
    writer.writerows(report_rows)

print(f'Cuerpos y diagnósticos: {bodies_path.relative_to(ROOT)}')
print(f'Reporte de balance: {report_path.relative_to(ROOT)}')
if deficits:
    print('No se creó labelstudio_tasks.csv porque faltan cuotas:', deficits)
else:
    labelstudio_path = output_path.parent / 'labelstudio_tasks.csv'
    with labelstudio_path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=LABEL_STUDIO_COLUMNS, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(balanced_rows)
    print(f'Tareas balanceadas listas para Label Studio: {len(balanced_rows)}')
    print(f'Archivo a importar: {labelstudio_path.relative_to(ROOT)}')

## 9. Importar solo si el balance está completo

Si la sección 8 crea `labelstudio_tasks.csv`, impórtalo en Label Studio como **List of tasks** con un formulario que use `$title` y `$body`. Si no se crea, revisa `balance_report.csv`: el corpus todavía no es equilibrado y no debe importarse.